                # 02 Feature Label Factory Lab

                Purpose: pull latest code, build leakage-aware features,
                validate feature availability timestamps, build labels after
                decision time, join a research dataset, and inspect summaries.
                


## 1. Mount Drive And Project Root


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")
Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)


## 2. Pull Latest Code


In [ ]:
import subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/umutergul74/btcusdt_quant_research.git"
GITHUB_BRANCH = "main"
project_path = Path(PROJECT_ROOT)

if (project_path / ".git").exists():
    subprocess.run(["git", "-C", PROJECT_ROOT, "remote", "set-url", "origin", GITHUB_REPO_URL], check=True)
    subprocess.run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)
else:
    visible_items = [p.name for p in project_path.iterdir() if not p.name.startswith(".")]
    if visible_items:
        raise RuntimeError(
            f"{PROJECT_ROOT} exists but is not a git checkout. "
            "Move existing Drive artifacts aside or clone the repository there before continuing. "
            f"Visible items: {visible_items[:10]}"
        )
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, PROJECT_ROOT], check=True)

if f"{PROJECT_ROOT}/src" not in sys.path:
    sys.path.append(f"{PROJECT_ROOT}/src")
print("Repository is ready at:", PROJECT_ROOT)


## 3. Install Dependencies And Stage Helper


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_ROOT}/requirements.txt"], check=True)

from pprint import pprint
from btc_quant.pipeline.stages import run_stage
from btc_quant.pipeline.stage_registry import list_stage_specs

RUN_ID = "debug_colab"

def run_stage_checked(stage_name, config_name=None):
    print(f"\n=== {stage_name} ===")
    result = run_stage(stage_name, config_name=config_name, project_root=PROJECT_ROOT, run_id=RUN_ID)
    pprint(result)
    if result.get("status") == "FAIL":
        raise RuntimeError(f"Stage failed: {stage_name}")
    return result


## 4. Review Feature And Label Configs


In [ ]:
                from pathlib import Path
                for rel in ["configs/features.yaml", "configs/labels.yaml"]:
                    path = Path(PROJECT_ROOT) / rel
                    print("\n---", rel, "---")
                    print(path.read_text())
                


## 5. Build Feature Store


In [ ]:
features_result = run_stage_checked("build_features", config_name="features")


## 6. Inspect Leakage Report


In [ ]:
                import json
                from pathlib import Path

                leakage_path = Path(PROJECT_ROOT) / "reports" / "experiments" / RUN_ID / "leakage_report.json"
                pprint(json.loads(leakage_path.read_text()))
                


## 7. Build Labels


In [ ]:
labels_result = run_stage_checked("build_labels", config_name="labels")


## 8. Inspect Label Summary


In [ ]:
                import json
                from pathlib import Path

                label_path = Path(PROJECT_ROOT) / "reports" / "experiments" / RUN_ID / "label_summary.json"
                pprint(json.loads(label_path.read_text()))
                


## 9. Build Research Dataset


In [ ]:
dataset_result = run_stage_checked("build_research_dataset")


## 10. Preview Research Dataset Columns


In [ ]:
                from pathlib import Path
                import pandas as pd

                dataset_path = Path(PROJECT_ROOT) / "data" / "processed" / "research_dataset.parquet"
                dataset = pd.read_parquet(dataset_path)
                print("Rows:", len(dataset))
                print("Columns:", len(dataset.columns))
                print(dataset[["bar_close_time", "decision_time", "label_start_time", "label_end_time", "target"]].head())
                


In [ ]:
# Auto-close the Colab runtime after every previous cell has completed.
# Set this to False before running if you want to keep the session alive for inspection.
AUTO_CLOSE_COLAB_RUNTIME = True

if AUTO_CLOSE_COLAB_RUNTIME:
    try:
        from google.colab import runtime
        print("All notebook cells completed. Releasing the Colab runtime now.")
        runtime.unassign()
    except Exception as exc:
        print(f"Auto-close skipped outside Colab or because runtime release failed: {exc}")
